# Части final_df: оптимизированная версия

Копия `spark_final_df_parts.ipynb` с теми же источниками. Отличия:

- таблица операций `ods_alpha.scd1_trx` читается один раз: секция 05a пишет промежуточную таблицу «договор и терминал», из неё считаются 05 и 05b;
- 04, 05 и MCC берут договоры из готового периметра 01, а не перечитывают озеро;
- периметр 01 — одна строка на договор, со сроками договора;
- нет ячейки Spark после 01, нет карты 08b и колонок, которых нет в дашборде;
- в срезе есть колонки `mvp_columns` из `final_df_script_3_1`: сроки договора, поля актуального тарифа, источник комиссии, флаги продуктов и `recommended_products`.

Запускать после `spark_final_df_parts.ipynb`: в начале её таблицы сохраняются под именами `..._prev`, в конце сравниваются с новыми.

Каждая часть пишет свою таблицу в `sandbox_ai`. В DRP ничего не пишется. Месячная комиссия берётся из DOC_OPER, аренда MPOS — отдельная колонка.


In [ ]:
import calendar
import logging
import warnings

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)

REPORT_MONTH = "2026-07"
year_i, month_i = (int(part) for part in REPORT_MONTH.split("-"))
month_start = f"{REPORT_MONTH}-01"
month_end = f"{REPORT_MONTH}-{calendar.monthrange(year_i, month_i)[1]:02d}"
next_y, next_m = (year_i + 1, 1) if month_i == 12 else (year_i, month_i + 1)
month_next_start = f"{next_y}-{next_m:02d}-01"
print("month", REPORT_MONTH, month_start, month_end, month_next_start)

In [ ]:
import getpass
import os

from rail_connectors.connection import connect

keytab_path = "/home/jovyan/test_requests/tech.keytab"
os.makedirs(os.path.dirname(keytab_path), exist_ok=True)

LAKE_USER = input("Логин Impala: ").strip()
LAKE_PASSWORD = getpass.getpass("Пароль Impala: ")
if not LAKE_USER or not LAKE_PASSWORD:
    raise RuntimeError("Логин и пароль нужны оба")

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": keytab_path,
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": LAKE_USER, "password": LAKE_PASSWORD},
)
imp._init_connection()
with imp:
    imp.execute("SELECT 1")
print("Impala ok")

In [ ]:
import time

from IPython.display import display

part_timings = {}

def save_part(table, select_sql, mem_limit="8g", allow_empty=False):
    started = time.perf_counter()
    imp._init_connection()
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        imp.execute(f"DROP TABLE IF EXISTS {table}")
        imp.execute(f"CREATE TABLE {table} STORED AS PARQUET AS\n{select_sql}")
        cnt = imp.fetch(f"SELECT count(*) AS n FROM {table}")
        preview = imp.fetch(f"SELECT * FROM {table} LIMIT 20")
    elapsed = round(time.perf_counter() - started, 1)
    part_timings[table] = elapsed
    print(table, f"{elapsed} сек")
    display(cnt)
    display(preview)
    n_rows = 0 if cnt is None or len(cnt) == 0 else int(cnt.iloc[0]["n"])
    if n_rows == 0 and not allow_empty:
        raise RuntimeError(f"Пустая таблица {table}")
    return n_rows

## Сохранить результаты старой тетрадки

Запускать после того, как `spark_final_df_parts.ipynb` дошла до среза месяца. Ячейка переименовывает её таблицы в `..._prev`, чтобы эта тетрадка их не стёрла. В конце они сравниваются с новыми.

Если `..._prev` уже есть, она не трогается: это повторный запуск, и старые результаты уже сохранены. Тогда текущая таблица с таким же именем просто пересчитается.


In [ ]:
PREV_SUFFIX = "_prev"
T_SLICE = "sandbox_ai.tmp_shestopalov_acq_month_slice"
PART_TABLES = [
    "sandbox_ai.tmp_shestopalov_acq_01_sa_perimeter",
    "sandbox_ai.tmp_shestopalov_acq_02_cdi_map",
    "sandbox_ai.tmp_shestopalov_acq_03_cft_map",
    "sandbox_ai.tmp_shestopalov_acq_05_trx_agr",
    "sandbox_ai.tmp_shestopalov_acq_05b_active_retl",
    "sandbox_ai.tmp_shestopalov_acq_mcc_by_agr",
    "sandbox_ai.tmp_shestopalov_acq_04_terminals",
    "sandbox_ai.tmp_shestopalov_acq_06_cft_attrs",
    "sandbox_ai.tmp_shestopalov_acq_09_tariff",
    "sandbox_ai.tmp_shestopalov_acq_10o_doc_oper",
    "sandbox_ai.tmp_shestopalov_acq_10m_mpos",
    "sandbox_ai.tmp_shestopalov_acq_10b_products",
    "sandbox_ai.tmp_shestopalov_acq_kedr_chod",
    T_SLICE,
]


def prev_name(full_name):
    return full_name + PREV_SUFFIX


def table_exists(full_name):
    db, name = full_name.split(".")
    imp._init_connection()
    with imp:
        found = imp.fetch(f"show tables in {db} like '{name}'")
    return found is not None and len(found) > 0


if not table_exists(T_SLICE) and not table_exists(prev_name(T_SLICE)):
    raise RuntimeError(
        "Нет среза старой тетрадки. Сначала доведите spark_final_df_parts.ipynb до последней ячейки."
    )

for table in PART_TABLES:
    saved = prev_name(table)
    if table_exists(saved):
        print("уже сохранена", saved)
    elif table_exists(table):
        imp._init_connection()
        with imp:
            imp.execute(f"ALTER TABLE {table} RENAME TO {saved}")
        print("сохранена", table, "->", saved)
    else:
        print("нет в старой тетрадке", table)

## 01. Периметр договоров SA

Договоры эквайринга, действующие в месяце, с терминалом типа P. Одна строка на договор: если у договора несколько версий в месяце, берётся самая поздняя.


In [ ]:
t01 = "sandbox_ai.tmp_shestopalov_acq_01_sa_perimeter"
sql01 = f"""
select
  n_agr,
  agr_id,
  n_cmp_client,
  contract_number,
  d_valid_from,
  d_valid_to,
  inn,
  company_name
from (
  select
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.c_agr_number as string) as contract_number,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
    cast(c.c_cmp_name as string) as company_name,
    row_number() over (
      partition by cast(a.n_agr as string)
      order by cast(a.d_valid_from as date) desc, cast(a.d_valid_to as date) desc
    ) as rn
  from ods_alpha.scd1_agreements a
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
    and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
    )
) x
where rn = 1
"""
save_part(t01, sql01)

## 02. CDI по ИНН из периметра


In [ ]:
t02 = "sandbox_ai.tmp_shestopalov_acq_02_cdi_map"
sql02 = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    trim(cast(soe.x_area_resp as string)) as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc, cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in (
      select inn from {t01}
    )
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""
save_part(t02, sql02)

## 03. CFT по CDI из предыдущей таблицы


In [ ]:
t03 = "sandbox_ai.tmp_shestopalov_acq_03_cft_map"
sql03 = f"""
select
  cast(e.party_id as string) as cdi_id,
  cast(e.cmo_ext_party_source_id as string) as cft_id
from cdiul.ext_id_org e
where cast(e.party_id as string) in (
    select cdi_id from {t02} where cdi_id is not null
  )
  and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
"""
save_part(t03, sql03)

## 05a. Операции по договору и терминалу

Единственное чтение `ods_alpha.scd1_trx`. Одна строка — договор и терминал за месяц. Самая долгая ячейка.


In [ ]:
t05a = "sandbox_ai.tmp_shestopalov_acq_05a_trx_agr_term"
sql05a = f"""
with sa_agr as (
  select n_agr, n_cmp_client, inn
  from {t01}
),
fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
trx_base_raw as (
  select cast(t.n_trx as string) as n_trx, cast(t.c_nter as string) as c_nter, cast(t.n_amt_src as double) as n_amt_src
  from ods_alpha.scd1_trx t
  join fiid_rshb fr
    on fr.c_fiid = cast(t.c_fiid_acq as string)
  where t.d_trx_orig >= '{month_start}'
    and t.d_trx_orig < '{month_next_start}'
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
),
trx_base as (
  select n_trx, max(c_nter) as c_nter, max(n_amt_src) as n_amt_src
  from trx_base_raw
  group by n_trx
),
ta_raw as (
  select
    cast(a.n_trx as string) as n_trx,
    cast(a.n_agr as string) as n_agr,
    coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax,
    case when coalesce(a.ods_deleted_flg, '0') <> '1' then 1 else 0 end as ta_alive
  from ods_alpha.scd1_trx_acq a
  join trx_base tb on tb.n_trx = cast(a.n_trx as string)
  join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
),
ta as (
  select n_trx, n_agr, max(n_amt_tax) as n_amt_tax, max(ta_alive) as ta_alive
  from ta_raw
  group by n_trx, n_agr
),
trx_keys as (
  select distinct n_trx
  from ta
),
trx_int_agg as (
  select cast(i.n_trx as string) as n_trx, max(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
  from ods_alpha.scd1_trx_int i
  join trx_keys k on k.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
  group by cast(i.n_trx as string)
)
select
  ta.n_agr,
  sa.n_cmp_client,
  sa.inn,
  tb.c_nter,
  count(distinct ta.n_trx) as trx_cnt,
  sum(tb.n_amt_src) as trx_sum,
  sum(ta.n_amt_tax) as commission_from_ops,
  sum(coalesce(i.n_amt_fee, 0.0)) as int_component,
  max(case when coalesce(tb.n_amt_src, 0.0) > 1 then 1 else 0 end) as has_amt_gt1,
  max(case when coalesce(tb.n_amt_src, 0.0) > 1 and ta.ta_alive = 1 then 1 else 0 end) as has_alive_amt_gt1
from ta
join trx_base tb on tb.n_trx = ta.n_trx
join sa_agr sa on sa.n_agr = ta.n_agr
left join trx_int_agg i on i.n_trx = ta.n_trx
group by ta.n_agr, sa.n_cmp_client, sa.inn, tb.c_nter
"""
save_part(t05a, sql05a, mem_limit="16g")

## 05. Операции по договору

Складывается из 05a, озеро не читает.


In [ ]:
t05 = "sandbox_ai.tmp_shestopalov_acq_05_trx_agr"
sql05 = f"""
with by_agr as (
  select
    n_agr,
    max(n_cmp_client) as n_cmp_client,
    max(inn) as inn,
    sum(trx_cnt) as trx_cnt,
    sum(trx_sum) as trx_sum,
    sum(commission_from_ops) as commission_from_ops,
    sum(int_component) as int_component,
    count(distinct case when has_amt_gt1 = 1 then c_nter end) as active_terms
  from {t05a}
  group by n_agr
),
by_client as (
  select
    n_cmp_client,
    count(distinct case when has_amt_gt1 = 1 then c_nter end) as active_term_cnt
  from {t05a}
  where n_cmp_client is not null
  group by n_cmp_client
)
select
  a.n_agr,
  a.inn,
  a.trx_cnt,
  a.trx_sum,
  a.commission_from_ops,
  a.int_component,
  a.n_cmp_client,
  c.active_term_cnt,
  a.active_terms
from by_agr a
left join by_client c on c.n_cmp_client = a.n_cmp_client
"""
save_part(t05, sql05, mem_limit="8g")

## Сверка 05 со старой версией

Суммы по всем договорам из старой таблицы `..._05_trx_agr_prev` и новой. Число операций, оборот, комиссия, интерчейндж и активные терминалы должны совпасть. Число договоров может быть чуть меньше: периметр теперь держит одну строку на договор.


In [ ]:
old05 = prev_name(t05)
if not table_exists(old05):
    print(f"Нет {old05}: сверка пропущена")
else:
    sql_cmp = f"""
    select 'old' as version,
      count(*) as agr_cnt,
      sum(trx_cnt) as trx_cnt,
      sum(trx_sum) as trx_sum,
      sum(commission_from_ops) as commission_from_ops,
      sum(int_component) as int_component,
      sum(active_terms) as active_terms
    from {old05}
    union all
    select 'new',
      count(*),
      sum(trx_cnt),
      sum(trx_sum),
      sum(commission_from_ops),
      sum(int_component),
      sum(active_terms)
    from {t05}
    """
    imp._init_connection()
    with imp:
        cmp05 = imp.fetch(sql_cmp)
    display(cmp05)
    old_row = cmp05.loc[cmp05["version"] == "old"].iloc[0]
    new_row = cmp05.loc[cmp05["version"] == "new"].iloc[0]
    for col in ["trx_cnt", "trx_sum", "commission_from_ops", "int_component", "active_terms"]:
        diff = float(new_row[col] or 0) - float(old_row[col] or 0)
        status = "совпадает" if abs(diff) < 1 else "РАСХОДИТСЯ"
        print(f"{col}: {status}, разница {diff:,.2f}")

## 05b. Активные точки

Терминалы с оборотом больше 1 берутся из 05a, точки — из `ods_alpha.scd1_pos_terminals`.


In [ ]:
table_name = "sandbox_ai.tmp_shestopalov_acq_05b_active_retl"
sql_part = f"""
with active_nter as (
  select distinct n_agr, c_nter
  from {t05a}
  where has_alive_amt_gt1 = 1
)
select
  an.n_agr,
  count(distinct cast(p.c_nmrc as string)) as active_retl_cnt
from active_nter an
join ods_alpha.scd1_pos_terminals p
  on cast(p.c_nter as string) = an.c_nter
where coalesce(p.ods_deleted_flg, '0') <> '1'
  and p.c_nmrc is not null
group by an.n_agr
"""
save_part(table_name, sql_part, mem_limit="8g", allow_empty=True)

## MCC по договору

Справочник мерчантов по договорам периметра.


In [ ]:
t_mcc = "sandbox_ai.tmp_shestopalov_acq_mcc_by_agr"
sql_mcc = f"""
with sa as (
  select agr_id, n_agr, n_cmp_client
  from {t01}
),
raw_mcc as (
  select
    s.n_agr,
    cast(m.n_mcc as string) as mcc,
    cast(m.c_mrc_profile as string) as mcc_profile
  from sa s
  left join ods_alpha.scd1_merchants m
    on cast(m.n_cmp as string) = s.n_cmp_client
  where m.n_mcc is not null or m.c_mrc_profile is not null
)
select
  n_agr,
  group_concat(distinct mcc, ',') as mcc,
  group_concat(distinct mcc_profile, ',') as mcc_profile,
  count(distinct mcc) as mcc_cnt
from raw_mcc
group by n_agr
"""
save_part(t_mcc, sql_mcc, mem_limit="4g")

## 04. Терминалы и амортизация

Договоры из периметра 01, амортизация из `sandbox_ai.shestopalov_terminal_amortization_model_jan_aug`.


In [ ]:
table_name = "sandbox_ai.tmp_shestopalov_acq_04_terminals"
sql_part = f'''
with sa_agr as (
  select n_agr, n_cmp_client
  from {t01}
),

terms_547 as (
  select distinct
    sa.n_agr,
    sa.n_cmp_client,
    cast(t.c_nmrc as string) as c_nmrc
  from sa_agr sa
  join ods_alpha.scd1_agr_terms t
    on cast(t.n_agr as string) = sa.n_agr
  join ods_alpha.scd1_merchants m
    on cast(m.c_nmrc as string) = cast(t.c_nmrc as string)
  where t.c_nmrc is not null
    and upper(coalesce(trim(cast(m.c_mrc_name as string)), '')) not like 'REZERVNYI TERMINAL%'
    and coalesce(upper(trim(cast(m.acq_class as string))), 'NA') <> 'SV'
    and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and coalesce(m.ods_deleted_flg, '0') <> '1'
),
pos_period as (
  select distinct
    cast(p.c_nmrc as string) as c_nmrc,
    cast(p.c_pos_serial as string) as c_pos_serial,
    cast(p.c_nter as string) as c_nter
  from ods_alpha.scd1_pos_terminals p
  where p.c_pos_serial is not null
    and cast(p.d_ter_install as date) is not null
    and cast(p.d_ter_install as date) < date_add(cast('{month_end}' as date), 1)
    and (p.d_ter_close is null or cast(p.d_ter_close as date) >= cast('{month_start}' as date))
    and coalesce(p.ods_deleted_flg, '0') <> '1'
),
term_active_547 as (
  select
    t.n_agr,
    t.n_cmp_client,
    t.c_nmrc,
    p.c_pos_serial,
    p.c_nter
  from terms_547 t
  left join pos_period p
    on p.c_nmrc = t.c_nmrc
),
retl_547 as (
  select n_agr, count(distinct c_nmrc) as retl_cnt_547
  from term_active_547
  group by n_agr
),
retl_with_term_547 as (
  select n_agr, count(distinct c_nmrc) as retl_with_term_cnt_547
  from term_active_547
  where c_pos_serial is not null or c_nter is not null
  group by n_agr
),
term_547 as (
  select n_agr, count(distinct c_pos_serial) as term_cnt_547
  from term_active_547
  group by n_agr
),
-- One physical device (serial) -> one owner n_agr -> one amort amount.
-- Avoids multi-agr and mid-month c_nter rename double-count.
amort_serial_owner_547 as (
  select n_agr, c_nter, c_pos_serial
  from (
    select
      cast(n_agr as string) as n_agr,
      cast(c_nter as string) as c_nter,
      cast(c_pos_serial as string) as c_pos_serial,
      row_number() over (
        partition by coalesce(
          nullif(trim(cast(c_pos_serial as string)), ''),
          concat('__NTER__', cast(c_nter as string))
        )
        order by
          cast(n_agr as string),
          cast(c_nter as string)
      ) as rn
    from term_active_547
    where c_nter is not null
  ) z
  where rn = 1
),
amort_547 as (
  select
    o.n_agr,
    sum(coalesce(cast(am.amortization_for_report_month as double), 0.0)) as amortization_547
  from amort_serial_owner_547 o
  left join sandbox_ai.shestopalov_terminal_amortization_model_jan_aug am
    on cast(am.c_nter as string) = o.c_nter
   and cast(am.snapshot_month_start as date) = cast('{month_start}' as date)
  group by o.n_agr
),

fallback_agrs as (
  select sa.n_agr, sa.n_cmp_client
  from sa_agr sa
  left join retl_547 r547 on r547.n_agr = sa.n_agr
  left join term_547 t547 on t547.n_agr = sa.n_agr
  where r547.n_agr is null and t547.n_agr is null
),
old_nmrc as (
  select fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string) as c_nmrc
  from fallback_agrs fa
  join ods_alpha.scd1_merchants mm
    on cast(mm.n_cmp as string) = fa.n_cmp_client
  where mm.c_nmrc is not null
    and coalesce(mm.ods_deleted_flg, '0') <> '1'
  group by fa.n_agr, fa.n_cmp_client, cast(mm.c_nmrc as string)
),
old_term_active as (
  select
    o.n_agr,
    o.n_cmp_client,
    o.c_nmrc,
    cast(t.c_nter as string) as c_nter,
    cast(t.c_pos_serial as string) as c_pos_serial
  from old_nmrc o
  join ods_alpha.scd1_pos_terminals t
    on cast(t.c_nmrc as string) = o.c_nmrc
  where t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and cast(t.d_ter_install as date) is not null
    and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
    and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
  group by o.n_agr, o.n_cmp_client, o.c_nmrc, cast(t.c_nter as string), cast(t.c_pos_serial as string)
),
old_retl as (
  select n_agr, count(distinct c_nmrc) as retl_cnt_old
  from old_term_active
  group by n_agr
),
old_term as (
  select n_agr, count(distinct c_nter) as term_cnt_old
  from old_term_active
  group by n_agr
),
amort_serial_owner_old as (
  select n_agr, c_nter, c_pos_serial
  from (
    select
      cast(n_agr as string) as n_agr,
      cast(c_nter as string) as c_nter,
      cast(c_pos_serial as string) as c_pos_serial,
      row_number() over (
        partition by coalesce(
          nullif(trim(cast(c_pos_serial as string)), ''),
          concat('__NTER__', cast(c_nter as string))
        )
        order by
          cast(n_agr as string),
          cast(c_nter as string)
      ) as rn
    from old_term_active
    where c_nter is not null
  ) z
  where rn = 1
),
old_amort as (
  select
    o.n_agr,
    sum(coalesce(cast(am.amortization_for_report_month as double), 0.0)) as amortization_old
  from amort_serial_owner_old o
  left join sandbox_ai.shestopalov_terminal_amortization_model_jan_aug am
    on cast(am.c_nter as string) = o.c_nter
   and cast(am.snapshot_month_start as date) = cast('{month_start}' as date)
  group by o.n_agr
)

select
  sa.n_agr,
  sa.n_cmp_client,
  coalesce(r547.retl_cnt_547, rold.retl_cnt_old) as retl_cnt,
  coalesce(rwt547.retl_with_term_cnt_547, rold.retl_cnt_old, 0) as retl_with_term_cnt,
  coalesce(t547.term_cnt_547, told.term_cnt_old) as term_cnt,
  coalesce(a547.amortization_547, aold.amortization_old, 0.0) as amortization,
  case
    when r547.n_agr is not null or t547.n_agr is not null then '547_like'
    when rold.n_agr is not null or told.n_agr is not null then 'fallback_old'
    else 'no_data'
  end as term_calc_source
from sa_agr sa
left join retl_547 r547 on r547.n_agr = sa.n_agr
left join retl_with_term_547 rwt547 on rwt547.n_agr = sa.n_agr
left join term_547 t547 on t547.n_agr = sa.n_agr
left join amort_547 a547 on a547.n_agr = sa.n_agr
left join old_retl rold on rold.n_agr = sa.n_agr
left join old_term told on told.n_agr = sa.n_agr
left join old_amort aold on aold.n_agr = sa.n_agr
'''
save_part(table_name, sql_part, mem_limit="16g", allow_empty=True)

## 06. Атрибуты ЦФТ

CFT из таблицы 03. Только колонки, которые читает срез.


In [ ]:
table_name = "sandbox_ai.tmp_shestopalov_acq_06_cft_attrs"
sql_part = f'''
with merchant_ranked as (
  select
    m.id as r2_id_raw,
    m.c_cl_org as cft_id_raw,
    m.c_depart as c_depart_raw,
    m.c_tariff_plan as c_tariff_plan_raw,
    row_number() over (
      partition by m.c_cl_org
      order by
        case when m.c_tariff_plan is not null then 0 else 1 end,
        case when m.c_depart is not null then 0 else 1 end,
        m.id desc
    ) as rn
  from ods.scd1_z_r2_ip_merchants m
  where cast(m.c_cl_org as string) in (select cft_id from sandbox_ai.tmp_shestopalov_acq_03_cft_map where cft_id is not null)
    and m.c_cl_org is not null
    and coalesce(m.ods_deleted_flg, '0') <> '1'
),
r2_raw as (
  select
    r2_id_raw,
    cft_id_raw,
    c_depart_raw,
    c_tariff_plan_raw
  from merchant_ranked
  where rn = 1
),
r2_enriched as (
  select
    r2.r2_id_raw as r2_id,
    r2.cft_id_raw as cft_id,
    r2.c_tariff_plan_raw as c_tariff_plan,
    corp.c_register_gos_reg_num_rec as ogrn,
    dep.c_name as vsp_name,
    dep.c_num as vsp_code,
    br.c_shortlabel as filial_rf_raw,
    tp.c_name as tariff_name_legacy
  from r2_raw r2
  left join ods.scd1_z_cl_corp corp on corp.id = r2.cft_id_raw
  left join ods.scd1_z_depart dep on dep.id = r2.c_depart_raw
  left join ods.scd1_z_branch br on br.id = dep.c_filial
  left join ods.scd1_z_r2_tariff_plan tp on tp.id = r2.c_tariff_plan_raw
)
select
  cft_id,
  ogrn,
  vsp_name,
  vsp_code,
  filial_rf_raw,
  tariff_name_legacy
from r2_enriched
'''
save_part(table_name, sql_part, mem_limit="8g", allow_empty=True)

## 09. Актуальный тариф договора

Остаётся самая свежая строка тарифа на месяц отчёта.


In [ ]:
table_name = "sandbox_ai.tmp_shestopalov_acq_09_tariff"
sql_part = f'''
with agr_actual as (
  select
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr_actual,
    cast(a.c_agr_number as string) as contract_number_acq,
    cast(a.d_valid_from as date) as d_valid_from_actual,
    cast(a.d_valid_to as date) as d_valid_to_actual,
    row_number() over (
      partition by cast(a.abs_agr_id as string)
      order by cast(a.d_valid_from as date) desc, cast(a.n_agr as string) desc
    ) as rn
  from ods_alpha.scd1_agreements a
  where cast(a.abs_agr_id as string) in (select agr_id from sandbox_ai.tmp_shestopalov_acq_01_sa_perimeter)
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_end}' as date))
    and coalesce(a.ods_deleted_flg, '0') <> '1'
),
merchant_one as (
  select
    cast(m.id as string) as agr_id,
    cast(m.c_tariff_plan as string) as c_tariff_plan,
    row_number() over (
      partition by cast(m.id as string)
      order by cast(m.c_tariff_plan as string) desc
    ) as rn
  from ods.scd1_z_r2_ip_merchants m
  where cast(m.id as string) in (select agr_id from sandbox_ai.tmp_shestopalov_acq_01_sa_perimeter)
    and coalesce(m.ods_deleted_flg, '0') <> '1'
)
select
  m.agr_id,
  aa.n_agr_actual,
  aa.contract_number_acq,
  aa.d_valid_from_actual,
  aa.d_valid_to_actual,
  m.c_tariff_plan,
  cast(tp.c_name as string) as tariff_name_actual
from merchant_one m
left join agr_actual aa
  on aa.agr_id = m.agr_id
 and aa.rn = 1
left join ods.scd1_z_r2_tariff_plan tp
  on cast(tp.id as string) = m.c_tariff_plan
where m.rn = 1
'''
save_part(table_name, sql_part, mem_limit="8g", allow_empty=True)

## 10o. Месячная комиссия DOC_OPER

`commission_monthly` равен `round(сумма c_calc_summ / 1.22, 2)`. Тариф `0` берёт документы следующего месяца, остальные тарифы — месяц отчёта. Если документов нет, в таблице стоит 0. Аренда MPOS сюда не подставляется.


In [ ]:
t_oper = "sandbox_ai.tmp_shestopalov_acq_10o_doc_oper"
t01 = "sandbox_ai.tmp_shestopalov_acq_01_sa_perimeter"
t09 = "sandbox_ai.tmp_shestopalov_acq_09_tariff"
sql_oper = f"""
with docs as (
  select
    cast(m.id as string) as agr_id,
    cast(o.c_date_create as timestamp) as c_date_create,
    cast(o.c_calc_summ as double) as c_calc_summ
  from ods.scd1_z_R2_IP_DOG_OPER o
  join ods.scd1_z_r2_ip_merchants m
    on m.id = o.c_parent_id
  join ods.scd1_z_client cl
    on m.c_cl_org = cl.id
  where o.c_parent_class = 'R2_IP_MERCHANTS'
    and cl.class_id = 'CL_ORG'
    and cast(o.c_date_create as date) >= cast('{month_start}' as date)
    and cast(o.c_date_create as date) < add_months(cast('{month_start}' as date), 2)
),
tariff as (
  select
    cast(agr_id as string) as agr_id,
    trim(cast(c_tariff_plan as string)) as c_tariff_plan
  from {t09}
),
perimeter as (
  select distinct cast(agr_id as string) as agr_id
  from {t01}
  where agr_id is not null
)
select
  p.agr_id,
  round(coalesce(sum(d.c_calc_summ), 0) / 1.22, 2) as commission_monthly,
  round(coalesce(sum(d.c_calc_summ), 0), 2) as commission_monthly_oper_raw,
  count(d.c_calc_summ) as oper_doc_rows
from perimeter p
left join tariff t
  on t.agr_id = p.agr_id
left join docs d
  on d.agr_id = p.agr_id
 and (
    (
      coalesce(t.c_tariff_plan, '') in ('0', '0.0', '0,0')
      and cast(d.c_date_create as date) >= add_months(cast('{month_start}' as date), 1)
      and cast(d.c_date_create as date) < add_months(cast('{month_start}' as date), 2)
    )
    or (
      coalesce(t.c_tariff_plan, '') not in ('0', '0.0', '0,0')
      and cast(d.c_date_create as date) >= cast('{month_start}' as date)
      and cast(d.c_date_create as date) < add_months(cast('{month_start}' as date), 1)
    )
  )
group by p.agr_id
"""
save_part(t_oper, sql_oper, mem_limit="8g", allow_empty=True)

## 10m. MPOS отдельно

Колонка `commission_monthly_mpos`. В месячную комиссию не подставляется.


In [ ]:
table_name = "sandbox_ai.tmp_shestopalov_acq_10m_mpos"
sql_part = f'''
with rent_base as (
  select
    cast(c_nmrc as string) as c_nmrc,
    cast(d_rent as date) as d_rent_dt,
    cast(n_amt as double) as n_amt_num,
    cast(ods_commit_ts as timestamp) as ods_commit_ts,
    cast(ods_insert_ts as timestamp) as ods_insert_ts,
    cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
    coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
  from ods_alpha.scd1_mrc_pos_rent
  where c_nmrc is not null
    and cast(d_rent as date) between cast('{month_start}' as date) and cast('{month_end}' as date)
),
rent_ranked as (
  select
    *,
    row_number() over (
      partition by c_nmrc, d_rent_dt
      order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
    ) as rn
  from rent_base
  where ods_deleted_flg not in ('1', 'Y', 'y')
),
rent_dedup as (
  select c_nmrc, d_rent_dt, n_amt_num
  from rent_ranked
  where rn = 1
),
terms_active as (
  select distinct
    cast(t.n_agr as string) as n_agr,
    cast(t.c_nmrc as string) as c_nmrc,
    cast(t.d_valid_from as date) as d_valid_from,
    cast(t.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agr_terms t
  where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and t.c_nmrc is not null
    and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
),
agreements_active as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agreements a
  where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
),
companies_active as (
  select distinct
    cast(c.n_cmp as string) as n_cmp,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
  from ods_alpha.scd1_companies c
  where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and c.c_inn is not null
),
mapped_raw as (
  select
    r.c_nmrc,
    r.d_rent_dt,
    r.n_amt_num,
    c.inn_key,
    cast(a.agr_id as string) as agr_id_key
  from rent_dedup r
  left join terms_active t
    on t.c_nmrc = r.c_nmrc
   and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
  left join agreements_active a
    on a.n_agr = t.n_agr
   and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
  left join companies_active c
    on c.n_cmp = a.n_cmp_client
),
map_stats as (
  select
    c_nmrc,
    d_rent_dt,
    n_amt_num,
    count(distinct case when inn_key is not null and agr_id_key is not null
      then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
  from mapped_raw
  group by c_nmrc, d_rent_dt, n_amt_num
),
mapped_unique as (
  select
    mr.c_nmrc,
    mr.d_rent_dt,
    mr.n_amt_num,
    max(mr.inn_key) as inn_key,
    max(mr.agr_id_key) as agr_id_key
  from mapped_raw mr
  join map_stats ms
    on ms.c_nmrc = mr.c_nmrc
   and ms.d_rent_dt = mr.d_rent_dt
   and ms.n_amt_num = mr.n_amt_num
  where ms.valid_key_cnt = 1
    and mr.inn_key is not null
    and mr.agr_id_key is not null
  group by mr.c_nmrc, mr.d_rent_dt, mr.n_amt_num
)
select
  inn_key as inn,
  agr_id_key as agr_id,
  sum(n_amt_num) as commission_monthly_mpos
from mapped_unique
group by inn_key, agr_id_key
'''
save_part(table_name, sql_part, mem_limit="8g", allow_empty=True)

## 10b. Продукты клиента

Длинный список: ИНН и название продукта на конец месяца.


In [ ]:
table_name = "sandbox_ai.tmp_shestopalov_acq_10b_products"
sql_part = f'''
WITH rko_dt AS (
  SELECT max(dt_part) AS dt_part
  FROM sandbox_ai.prod__kuznetsov_lle__acc_ul_trx_features_impala
  WHERE dt_part <= date '{month_end}'
),
rko_inns AS (
  SELECT DISTINCT
    regexp_replace(trim(cast(f.inn AS string)), '[^0-9]', '') AS inn
  FROM sandbox_ai.prod__kuznetsov_lle__acc_ul_trx_features_impala f
  CROSS JOIN rko_dt d
  WHERE f.dt_part = d.dt_part
    AND f.inn IS NOT NULL
),
current_agrs AS (
  SELECT
    regexp_replace(trim(cast(inn AS string)), '[^0-9]', '') AS inn,
    lower(trim(cast(target_name AS string))) AS product_name,
    1 AS has_product
  FROM sandbox_ai.le_nbo_scoring__targets_product_agreems
  WHERE c_date_begin <= '{month_end}'
    AND (c_date_close > '{month_end}' OR c_date_close IS NULL)
    AND inn IS NOT NULL
    AND target_name IS NOT NULL
  GROUP BY 1, 2
),
union_flags AS (
  SELECT inn, product_name, has_product
  FROM current_agrs

  UNION ALL

  SELECT inn, 'rko' AS product_name, 1 AS has_product
  FROM rko_inns
)
SELECT
  inn,
  product_name,
  max(has_product) AS has_product
FROM union_flags
WHERE inn IS NOT NULL
  AND inn <> ''
GROUP BY inn, product_name
'''
save_part(table_name, sql_part, mem_limit="8g", allow_empty=True)

## ЧОД Кедра

Своя таблица по ИНН и месяцу.


In [ ]:
t_kedr = "sandbox_ai.tmp_shestopalov_acq_kedr_chod"
yearmm = int(REPORT_MONTH.replace("-", ""))
sql_kedr = f'''
select
  inn,
  max(yearmm) as yearmm,
  sum(kedr_obshiy_chod) as kedr_obshiy_chod,
  sum(src_cnt) as src_cnt
from (
  select
    regexp_replace(trim(cast(inn as string)), '[^0-9]', '') as inn,
    cast(yearmm as bigint) as yearmm,
    cast(kedr_obshiy_chod as double) as kedr_obshiy_chod,
    cast(src_cnt as bigint) as src_cnt
  from sandbox_ai.shestopalov_kedr_obshiy_chod_inn_month
  where cast(yearmm as bigint) = {yearmm}
) k
where inn is not null and inn <> ''
group by inn
'''
save_part(t_kedr, sql_kedr, mem_limit="4g", allow_empty=True)

## Срез договора за месяц

Одна строка — один договор. Колонки как в `mvp_columns` из `final_df_script_3_1`, плюс флаги продуктов и `recommended_products`. `commission_monthly` из DOC_OPER, `commission_monthly_mpos` — отдельно.


In [ ]:
t_slice = "sandbox_ai.tmp_shestopalov_acq_month_slice"
sql_slice = f"""
with cdi_one as (
  select inn, cdi_id, ssp_ocrm
  from (
    select
      inn,
      cdi_id,
      ssp_ocrm,
      row_number() over (partition by inn order by cdi_id) as rn
    from sandbox_ai.tmp_shestopalov_acq_02_cdi_map
  ) x
  where rn = 1
),
cft_one as (
  select cdi_id, cft_id
  from (
    select
      cdi_id,
      cft_id,
      row_number() over (partition by cdi_id order by cft_id) as rn
    from sandbox_ai.tmp_shestopalov_acq_03_cft_map
  ) x
  where rn = 1
),
mpos_one as (
  select
    cast(agr_id as string) as agr_id,
    inn,
    sum(commission_monthly_mpos) as commission_monthly_mpos
  from sandbox_ai.tmp_shestopalov_acq_10m_mpos
  group by cast(agr_id as string), inn
),
prod_flags as (
  select
    inn,
    max(case when product_name = 'rko' then coalesce(has_product, 0) else 0 end) as rko,
    max(case when product_name = 'accounting' then coalesce(has_product, 0) else 0 end) as accounting,
    max(case when product_name = 'business_cards' then coalesce(has_product, 0) else 0 end) as business_cards,
    max(case when product_name = 'credit' then coalesce(has_product, 0) else 0 end) as credit,
    max(case when product_name = 'dbo' then coalesce(has_product, 0) else 0 end) as dbo,
    max(case when product_name = 'deposit' then coalesce(has_product, 0) else 0 end) as deposit,
    max(case when product_name = 'insurance' then coalesce(has_product, 0) else 0 end) as insurance,
    max(case when product_name = 'loyalty_program' then coalesce(has_product, 0) else 0 end) as loyalty_program,
    max(case when product_name = 'nmo' then coalesce(has_product, 0) else 0 end) as nmo,
    max(case when product_name = 'nso' then coalesce(has_product, 0) else 0 end) as nso,
    max(case when product_name = 'pravocard' then coalesce(has_product, 0) else 0 end) as pravocard,
    max(case when product_name = 'salary_project' then coalesce(has_product, 0) else 0 end) as salary_project,
    max(case when product_name = 'self_inkass' then coalesce(has_product, 0) else 0 end) as self_inkass,
    max(case when product_name = 'service_package' then coalesce(has_product, 0) else 0 end) as service_package,
    max(case when product_name = 'sms_info' then coalesce(has_product, 0) else 0 end) as sms_info
  from sandbox_ai.tmp_shestopalov_acq_10b_products
  group by inn
),
base as (
  select
    '{REPORT_MONTH}' as report_month,
    '{month_start}' as snapshot_month_start,
    p.inn,
    p.company_name,
    p.agr_id,
    p.n_agr,
    p.contract_number,
    p.d_valid_from,
    p.d_valid_to,
    tar.n_agr_actual,
    tar.contract_number_acq,
    tar.d_valid_from_actual,
    tar.d_valid_to_actual,
    cdi.cdi_id,
    cdi.ssp_ocrm,
    cft.cft_id,
    r2.ogrn,
    r2.filial_rf_raw as filial_rf,
    r2.vsp_name,
    r2.vsp_code,
    coalesce(
      nullif(trim(cast(tar.tariff_name_actual as string)), ''),
      cast(r2.tariff_name_legacy as string)
    ) as tariff_name,
    case
      when nullif(trim(cast(tar.tariff_name_actual as string)), '') is not null then 'agr_id_actual'
      else 'legacy_cft'
    end as tariff_source,
    tar.c_tariff_plan,
    term.retl_cnt,
    least(
      greatest(coalesce(term.retl_with_term_cnt, 0), 0),
      coalesce(term.retl_cnt, 0)
    ) as retl_with_term_cnt,
    retl.active_retl_cnt,
    term.term_cnt,
    trx.active_terms,
    trx.active_term_cnt,
    trx.trx_cnt,
    trx.trx_sum,
    trx.commission_from_ops,
    coalesce(oper.commission_monthly, 0) as commission_monthly,
    case
      when coalesce(oper.oper_doc_rows, 0) > 0 then 'oper_doc_net'
      else 'oper_doc_missing'
    end as commission_monthly_source,
    mpos.commission_monthly_mpos,
    coalesce(oper.commission_monthly_oper_raw, 0) as commission_monthly_oper_raw,
    coalesce(oper.commission_monthly, 0) as commission_monthly_oper_net,
    trx.int_component,
    term.amortization,
    term.term_calc_source,
    mcc.mcc,
    mcc.mcc_profile,
    mcc.mcc_cnt,
    coalesce(prod.rko, 0) as rko,
    coalesce(prod.accounting, 0) as accounting,
    coalesce(prod.business_cards, 0) as business_cards,
    coalesce(prod.credit, 0) as credit,
    coalesce(prod.dbo, 0) as dbo,
    coalesce(prod.deposit, 0) as deposit,
    coalesce(prod.insurance, 0) as insurance,
    coalesce(prod.loyalty_program, 0) as loyalty_program,
    coalesce(prod.nmo, 0) as nmo,
    coalesce(prod.nso, 0) as nso,
    coalesce(prod.pravocard, 0) as pravocard,
    coalesce(prod.salary_project, 0) as salary_project,
    coalesce(prod.self_inkass, 0) as self_inkass,
    coalesce(prod.service_package, 0) as service_package,
    coalesce(prod.sms_info, 0) as sms_info,
    kedr.kedr_obshiy_chod
  from {t01} p
  left join cdi_one cdi on cdi.inn = p.inn
  left join cft_one cft on cft.cdi_id = cdi.cdi_id
  left join sandbox_ai.tmp_shestopalov_acq_04_terminals term on term.n_agr = p.n_agr
  left join sandbox_ai.tmp_shestopalov_acq_05_trx_agr trx on trx.n_agr = p.n_agr
  left join sandbox_ai.tmp_shestopalov_acq_05b_active_retl retl on retl.n_agr = p.n_agr
  left join sandbox_ai.tmp_shestopalov_acq_06_cft_attrs r2
    on cast(r2.cft_id as string) = cft.cft_id
  left join sandbox_ai.tmp_shestopalov_acq_09_tariff tar on tar.agr_id = p.agr_id
  left join sandbox_ai.tmp_shestopalov_acq_10o_doc_oper oper on oper.agr_id = p.agr_id
  left join mpos_one mpos on mpos.agr_id = p.agr_id and mpos.inn = p.inn
  left join sandbox_ai.tmp_shestopalov_acq_mcc_by_agr mcc on mcc.n_agr = p.n_agr
  left join prod_flags prod on prod.inn = p.inn
  left join sandbox_ai.tmp_shestopalov_acq_kedr_chod kedr on kedr.inn = p.inn
),
calc as (
  select
    base.*,
    case
      when lower(coalesce(tariff_name, '')) like '%акт%' then 'По актам'
      when lower(coalesce(tariff_name, '')) like '%акцион%'
        or lower(coalesce(tariff_name, '')) like '%меню%'
        or lower(coalesce(tariff_name, '')) like '%сезон%' then 'Акционный'
      when lower(coalesce(tariff_name, '')) like '%индив%' then 'Индивидуальный'
      when lower(coalesce(tariff_name, '')) like '%стандарт%' then 'Стандарт'
      else 'Не сегментирован'
    end as tariff_short,
    retl_with_term_cnt * 1926 as aur,
    coalesce(commission_from_ops, 0) + commission_monthly as commission_total,
    coalesce(commission_from_ops, 0) + commission_monthly + coalesce(int_component, 0) as chod,
    regexp_replace(
      concat(
      case when rko = 1 then '' else 'РКО, ' end,
      case when business_cards = 1 then '' else 'Бизнес-карта, ' end,
      case when dbo = 1 then '' else 'ДБО, ' end,
      case when insurance = 1 then '' else 'Страхование, ' end,
      case when pravocard = 1 then '' else 'Правокард, ' end,
      case when salary_project = 1 then '' else 'Зарплатный проект, ' end,
      case when self_inkass = 1 then '' else 'Самоинкассация, ' end,
      case when sms_info = 1 then '' else 'СМС-инфо, ' end
      ),
      ', $',
      ''
    ) as recommended_products
  from base
)
select
  calc.*,
  chod - aur - coalesce(amortization, 0) as fin_result,
  case when chod - aur - coalesce(amortization, 0) > 0 then 1 else 0 end as tsp_effective,
  coalesce(commission_from_ops, 0) / nullif(trx_sum, 0) * 100 as acq_pct,
  chod / nullif(trx_sum, 0) * 100 as chod_pct
from calc
"""
n_slice = save_part(t_slice, sql_slice, mem_limit="8g")
imp._init_connection()
with imp:
    n_per = int(imp.fetch(f"select count(*) as n from {t01}").iloc[0]["n"])
    n_agr_uniq = int(imp.fetch(
        f"select count(distinct n_agr) as n from {t_slice}"
    ).iloc[0]["n"])
print("slice", n_slice, "perimeter", n_per, "unique n_agr", n_agr_uniq)
if n_slice != n_per or n_agr_uniq != n_slice:
    raise RuntimeError(
        f"Срез {n_slice} строк, периметр {n_per}, договоров {n_agr_uniq}. Склейка размножила договоры."
    )

## Spark читает срез

Локально, без YARN. Открывает файлы готового среза.


In [ ]:
import os
from pyspark.sql import SparkSession

log4j_path = "/tmp/acq_spark_log4j2.properties"
with open(log4j_path, "w", encoding="utf-8") as log4j_file:
    log4j_file.write("\n".join([
        "status = error",
        "name = AcqSpark",
        "appender.console.type = Console",
        "appender.console.name = console",
        "appender.console.target = SYSTEM_ERR",
        "appender.console.layout.type = PatternLayout",
        "appender.console.layout.pattern = %p %c{1}: %m%n",
        "rootLogger.level = error",
        "rootLogger.appenderRef.stdout.ref = console",
    ]))
os.environ["SPARK_SUBMIT_OPTS"] = f"-Dlog4j2.configurationFile=file:{log4j_path}"

try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder
    .appName("acq-month-slice-local")
    .master("local[*]")
    .config("spark.log.level", "ERROR")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.parquet.binaryAsString", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

imp._init_connection()
with imp:
    desc = imp.fetch(f"DESCRIBE FORMATTED {t_slice}")
name_col, type_col = desc.columns[0], desc.columns[1]
mask = desc[name_col].astype(str).str.contains("Location", case=False, na=False)
location = str(desc.loc[mask, type_col].iloc[0]).strip()
print("location", location)

jvm = spark.sparkContext._jvm
conf = spark.sparkContext._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(jvm.java.net.URI.create(location), conf)
children = list(fs.listStatus(jvm.org.apache.hadoop.fs.Path(location)))
for child in children:
    kind = "dir" if child.isDirectory() else "file"
    print(kind, child.getPath().getName(), child.getLen())
files = [str(c.getPath()) for c in children if c.isFile() and c.getLen() > 0]
dirs = [
    str(c.getPath())
    for c in children
    if c.isDirectory() and not c.getPath().getName().startswith(("_", "."))
]
data_path = files[0] if files else dirs[0]
print("read", data_path)
part = spark.read.option("recursiveFileLookup", "true").parquet(data_path)
print("rows ok", part.count())

## Сравнение со старой тетрадкой

Число строк по каждой части, активные точки по договорам и итоговые суммы среза. Ниже время каждой части в этой тетрадке. Время старой тетрадки смотрите на странице запросов Impala, в колонке длительности у её `CREATE TABLE`.


In [ ]:
import pandas as pd

rows = []
for table in PART_TABLES:
    saved = prev_name(table)
    old_n = None
    new_n = None
    has_old = table_exists(saved)
    has_new = table_exists(table)
    imp._init_connection()
    with imp:
        if has_old:
            old_n = int(imp.fetch(f"select count(*) as n from {saved}").iloc[0]["n"])
        if has_new:
            new_n = int(imp.fetch(f"select count(*) as n from {table}").iloc[0]["n"])
    rows.append({"table": table.split(".")[1], "old_rows": old_n, "new_rows": new_n})
rows_cmp = pd.DataFrame(rows)
display(rows_cmp)

In [ ]:
old05b = prev_name("sandbox_ai.tmp_shestopalov_acq_05b_active_retl")
new05b = "sandbox_ai.tmp_shestopalov_acq_05b_active_retl"
if not table_exists(old05b):
    print(f"Нет {old05b}: сверка активных точек пропущена")
else:
    sql_retl = f"""
    select
      count(*) as agr_cnt,
      sum(case when o.n_agr is null then 1 else 0 end) as only_new,
      sum(case when n.n_agr is null then 1 else 0 end) as only_old,
      sum(case when o.n_agr is not null and n.n_agr is not null
               and o.active_retl_cnt <> n.active_retl_cnt then 1 else 0 end) as diff_cnt
    from {old05b} o
    full join {new05b} n on n.n_agr = o.n_agr
    """
    imp._init_connection()
    with imp:
        retl_cmp = imp.fetch(sql_retl)
    display(retl_cmp)
    r = retl_cmp.iloc[0]
    if int(r["diff_cnt"] or 0) == 0:
        print("Активные точки: у общих договоров совпадают")
    else:
        print(f"Активные точки: расходятся у {int(r['diff_cnt'])} договоров")
    print("только в старой:", int(r["only_old"] or 0), "| только в новой:", int(r["only_new"] or 0))

In [ ]:
old_slice = prev_name(T_SLICE)
if not table_exists(old_slice):
    print(f"Нет {old_slice}: сверка среза пропущена")
else:
    metric_cols = ["trx_cnt", "trx_sum", "commission_from_ops", "commission_monthly",
                   "int_component", "amortization", "chod", "fin_result"]
    sums_sql = ", ".join(f"sum({c}) as {c}" for c in metric_cols)
    sql_slice_cmp = f"""
    select 'old' as version, count(*) as rows_cnt, count(distinct n_agr) as agr_cnt, {sums_sql}
    from {old_slice}
    union all
    select 'new', count(*), count(distinct n_agr), {sums_sql}
    from {T_SLICE}
    """
    imp._init_connection()
    with imp:
        slice_cmp = imp.fetch(sql_slice_cmp)
    display(slice_cmp)
    o = slice_cmp.loc[slice_cmp["version"] == "old"].iloc[0]
    n = slice_cmp.loc[slice_cmp["version"] == "new"].iloc[0]
    print("строк в старом срезе:", int(o["rows_cnt"]), "договоров:", int(o["agr_cnt"]))
    print("строк в новом срезе:", int(n["rows_cnt"]), "договоров:", int(n["agr_cnt"]))
    if int(n["rows_cnt"]) != int(n["agr_cnt"]):
        print("Новый срез: договор встречается больше одного раза")
    for col in metric_cols:
        old_v = float(o[col] or 0)
        new_v = float(n[col] or 0)
        pct = (new_v - old_v) / abs(old_v) * 100 if old_v else 0.0
        print(f"{col}: старый {old_v:,.2f} | новый {new_v:,.2f} | разница {pct:.3f}%")

In [ ]:
timings = (
    pd.DataFrame([{"table": t.split(".")[1], "seconds": s} for t, s in part_timings.items()])
    .sort_values("seconds", ascending=False)
)
display(timings)
print("всего минут:", round(timings["seconds"].sum() / 60, 1))